In [ ]:
!pip install groq openai
from groq import Groq
from openai import OpenAI
from google.colab import userdata, drive
import json, re, time, os

drive.mount('/content/drive')
drive_repo_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data'

client_groq = Groq(api_key=userdata.get('GROQ_API'))
client_openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_KEY')
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.5 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import re
import json

def harvest_json_robust(text):
    """
    Individually finds every {...} block in the text and parses them into a flat list.
    This is extremely robust against model formatting errors.
    """
    if not text: return []

    # 1. Clean the text of invisible control characters
    clean_text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)

    # 2. Find ALL strings that look like JSON objects {...}
    # This finds each pair's prediction separately
    matches = re.findall(r'\{[^{}]*\}', clean_text)

    harvested_list = []
    for m in matches:
        try:
            # Normalize common LLM typos
            m_clean = m.replace('TRUE', '"TRUE"').replace('FALSE', '"FALSE"').replace('PROBABLE', '"PROBABLE"')
            m_clean = m_clean.replace('""', '"') # Fix double quotes

            obj = json.loads(m_clean)
            if isinstance(obj, dict) and ("at" in obj or "isAt" in obj):
                harvested_list.append(obj)
        except:
            continue # Skip junk blocks

    return harvested_list

In [ ]:
from tqdm import tqdm
import json, re, time

EXAMPLES_4_SHOT = """
Example 1: Regional/City Containment
Text: "Die nationale Gesellschaft belgischer Eisenbahnen hielt gestern in Brüssel ihre erste Generalversammlung ab."
Pair: Person: Eisenbahnminister Lippens | Place: Belgien
Output: {"results": [{"at": "TRUE", "isAt": "TRUE"}]}

Example 2: Clear Physical Presence
Text: "Am Nachmittag wurde Herr Müller auf dem Marktplatz in Leipzig gesehen."
Pair: Person: Herr Müller | Place: Leipzig
Output: {"results": [{"at": "TRUE", "isAt": "TRUE"}]}

Example 3: Professional Origin / Affiliation (Not Present)
Text: "Der Gesandte von Österreich traf gestern Abend verspätet am Pariser Bahnhof ein."
Pair: Person: Gesandte | Place: Österreich
Output: {"results": [{"at": "TRUE", "isAt": "FALSE"}]}

Example 4: Total Lack of Spatial Connection
Text: "Kaiser Wilhelm reiste nach Berlin, während General Bonaparte Truppen bei Paris sammelte."
Pair: Person: General Bonaparte | Place: Berlin
Output: {"results": [{"at": "FALSE", "isAt": "FALSE"}]}

"""

def call_with_retry(client, model, messages, max_tokens):
    while True:
        try:
            return client.chat.completions.create(model=model, messages=messages, max_tokens=max_tokens, temperature=0.0)
        except Exception as e:
            if "429" in str(e) or "503" in str(e) or "rate" in str(e).lower() or "capacity" in str(e).lower():
                print("Rate limit or capacity issue, waiting 30 seconds...")
                time.sleep(30)
            else:
                raise e

In [ ]:
def find_best_examples_word(query_text, train_docs, n=4):
    query_words = set(query_text[:500].lower().split())
    scores = []
    for doc in train_docs:
        doc_words = set(doc["text"][:500].lower().split())
        intersection = len(query_words & doc_words)
        union = len(query_words | doc_words)
        score = intersection / union if union > 0 else 0
        scores.append((score, doc))
    scores.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scores[:n]]

In [ ]:
!pip install pylcs

import pylcs

def find_best_examples(query_text, train_docs, n=4):
    scores = []
    for doc in train_docs:
        score = pylcs.lcs_sequence_length(query_text[:500], doc["text"][:500])
        scores.append((score, doc))
    scores.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scores[:n]]

def format_retrieved_examples(example_docs):
    examples_str = ""
    for i, doc in enumerate(example_docs):
        examples_str += f"EXAMPLE {i+1}:\n"
        examples_str += f'Text: "{doc["text"][:300]}"\n'
        examples_str += "Pairs:\n"
        for j, pair in enumerate(doc["sampled_pairs"][:3]):  # max 3 pairs per example
            person = pair["pers_mentions_list"][0]
            place = pair["loc_mentions_list"][0]
            examples_str += f"{j}. Person: {person} | Place: {place}\n"
            examples_str += f'Output: {{"at": "{pair["at"]}", "isAt": "{pair["isAt"]}"}}\n'
        examples_str += "\n"
    return examples_str

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for pylcs: filename=pylcs-0.1.1-cp312-cp312-linux_x86_64.whl size=1345555 sha256=6c2ed0140d50a6e621d18ebb24c9308ffb63a6039c659bb8638614e407b0fd94
  Stored in directory: /root/.cache/pip/wheels/45/12/c0/770a9a5efece91c7f9d545590a2ce2e8bddbe866dada392e1c
Successfully built pylcs


In [ ]:
def run_hipe_experiment(docs, client, model_name, output_filename,
                        use_few_shot=True, use_retrieval=False,
                        train_docs=None, retrieval_method="lcs"):
    print(f"🚀 Starting Extraction: {output_filename}")
    results = []
    total_pairs = sum(len(doc["sampled_pairs"]) for doc in docs)
    processed = 0

    for doc in docs:
        doc_copy = json.loads(json.dumps(doc))

        pairs_list_str = "\n".join([
            f"{i}. Person: {p['pers_mentions_list'][0]} | Place: {p['loc_mentions_list'][0]}"
            for i, p in enumerate(doc_copy['sampled_pairs'])
        ])

        # build examples block
        if use_retrieval and train_docs:
            if retrieval_method == "word":
                retrieved = find_best_examples_word(doc_copy["text"], train_docs)
            else:
                retrieved = find_best_examples(doc_copy["text"], train_docs)
            examples_block = f"""
---
HERE ARE RETRIEVED EXAMPLES FROM SIMILAR HISTORICAL DOCUMENTS:
{format_retrieved_examples(retrieved)}
---
"""
        elif use_few_shot:
            examples_block = f"""
---
HERE ARE REFERENCE EXAMPLES TO EMULATE SYSTEM REASONING:
{EXAMPLES_4_SHOT}
---
"""
        else:
            examples_block = ""

        # --- AT PROMPT ---
        at_prompt = f"""You are an expert computational historian specializing in relation extraction.
TASK: Determine the historical relation 'at' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'at' represents a permanent, structural, institutional, professional, or residency-based geographic connection over time.
2. Use 'PROBABLE' for 'at' when strong contextual, regional, or family/professional affiliation implies geographic connectivity without explicit absolute proof.
3. Use 'TRUE' ONLY if there is OVERWHELMING and EXPLICIT evidence — the text directly states the person lives there, works there permanently, or was born there. A single visit or passing mention is NOT enough for TRUE.
4. Use 'FALSE' if no evidence is present or the context contradicts such a relation.

{examples_block}

TARGET TEXT FOR ANALYSIS:
"{doc_copy['text']}"

Evaluate the 'at' relationship for the following requested pairs exactly in sequence.
Return a JSON object with key "results" containing one object per pair: {{"at": "TRUE/PROBABLE/FALSE"}}.

PAIRS TO EVALUATE:
{pairs_list_str}"""

        # --- ISAT PROMPT ---
        isat_prompt = f"""You are an expert computational historian specializing in relation extraction.
TASK: Determine the temporal relation 'isAt' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'isAt' represents literal immediate physical presence at that place within the narrative moment.
2. 'isAt' is TRUE if there is evidence the person was at the location up to about one month before the publication date.
3. Use 'FALSE' if the person is elsewhere, the event happened in the distant past, or no evidence of current presence exists.

{examples_block}

TARGET TEXT FOR ANALYSIS:
"{doc_copy['text']}"

Evaluate the 'isAt' relationship for the following requested pairs exactly in sequence.
Return a JSON object with key "results" containing one object per pair: {{"isAt": "TRUE/FALSE"}}.

PAIRS TO EVALUATE:
{pairs_list_str}"""

        # --- AT CALL ---
        at_response = call_with_retry(client, model_name, [
            {"role": "user", "content": at_prompt}
        ], 2000)

        raw_at = at_response.choices[0].message.content
        if raw_at is None:
            raw_at = ""
        raw_at = re.sub(r'<think>.*?</think>', '', raw_at, flags=re.DOTALL).strip()
        at_preds = harvest_json_robust(raw_at)

        # --- ISAT CALL ---
        isat_response = call_with_retry(client, model_name, [
            {"role": "user", "content": isat_prompt}
        ], 2000)

        raw_isat = isat_response.choices[0].message.content
        if raw_isat is None:
            raw_isat = ""
        raw_isat = re.sub(r'<think>.*?</think>', '', raw_isat, flags=re.DOTALL).strip()
        isat_preds = harvest_json_robust(raw_isat)

        # --- MERGE ---
        for i, pair in enumerate(doc_copy['sampled_pairs']):
            gold_at, gold_isat = pair["at"], pair["isAt"]

            at_pred = at_preds[i] if i < len(at_preds) else {"at": "FALSE"}
            if isinstance(at_pred, list) and len(at_pred) > 0:
                at_pred = at_pred[0]
            if not isinstance(at_pred, dict):
                at_pred = {"at": "FALSE"}

            isat_pred = isat_preds[i] if i < len(isat_preds) else {"isAt": "FALSE"}
            if isinstance(isat_pred, list) and len(isat_pred) > 0:
                isat_pred = isat_pred[0]
            if not isinstance(isat_pred, dict):
                isat_pred = {"isAt": "FALSE"}

            at_val = str(at_pred.get("at", "FALSE")).upper()
            isAt_val = str(isat_pred.get("isAt", "FALSE")).upper()

            pair['at'] = "TRUE" if "TRUE" in at_val else ("PROBABLE" if "PROBABLE" in at_val else "FALSE")
            pair['isAt'] = "TRUE" if "TRUE" in isAt_val else "FALSE"
            if pair['at'] == "FALSE":
                pair['isAt'] = "FALSE"

            processed += 1
            print(f"[{processed}/{total_pairs}] pred_at={pair['at']} gold_at={gold_at} | pred_isAt={pair['isAt']} gold_isAt={gold_isat}")

        results.append(doc_copy)

        with open(output_filename, "a", encoding="utf-8") as f:
            f.write(json.dumps(doc_copy) + "\n")

        time.sleep(0.3)

    print(f"✅ Done. Saved locally to: {output_filename}")

In [ ]:
def load_hipe_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

cleaned_dev = {
    "en": load_hipe_jsonl(f"{drive_repo_path}/data/sandbox/en-dev-cleaned.jsonl"),
    "fr": load_hipe_jsonl(f"{drive_repo_path}/data/sandbox/fr-dev-cleaned.jsonl"),
    "de": load_hipe_jsonl(f"{drive_repo_path}/data/sandbox/de-dev-cleaned.jsonl"),
}

raw_dev = {
    "en": load_hipe_jsonl(f"{drive_repo_path}/data/sandbox/en-dev.jsonl"),
    "fr": load_hipe_jsonl(f"{drive_repo_path}/data/sandbox/fr-dev.jsonl"),
    "de": load_hipe_jsonl(f"{drive_repo_path}/data/sandbox/de-dev.jsonl"),
}

train_docs = {
    "en": load_hipe_jsonl(f"{drive_repo_path}/data/sandbox/en-train-cleaned.jsonl"),
    "fr": load_hipe_jsonl(f"{drive_repo_path}/data/sandbox/fr-train-cleaned.jsonl"),
    "de": load_hipe_jsonl(f"{drive_repo_path}/data/sandbox/de-train-cleaned.jsonl"),
}



In [ ]:
# === EDIT THESE FOR EACH RUN ===
client = client_openrouter
model_name = "qwen/qwen3-32b"
lang = "fr"
use_few_shot = False
use_retrieval = True
retrieval_method = "word"  # "lcs" or "word"
# ================================

tag = model_name.split("/")[-1].replace("-", "_")
if use_retrieval:
    shot = f"RETRIEVAL_{retrieval_method.upper()}"
elif use_few_shot:
    shot = "4SHOT"
else:
    shot = "ZEROSHOT"

out = f"qwen_{tag}_{shot}_{lang}.jsonl"

run_hipe_experiment(
    cleaned_dev[lang], client, model_name, out,
    use_few_shot=use_few_shot,
    use_retrieval=use_retrieval,
    train_docs=train_docs[lang],
    retrieval_method=retrieval_method
)


🚀 Starting Extraction: qwen_qwen3_32b_RETRIEVAL_WORD_fr.jsonl
[1/1498] pred_at=FALSE gold_at=FALSE | pred_isAt=FALSE gold_isAt=FALSE
[2/1498] pred_at=FALSE gold_at=TRUE | pred_isAt=FALSE gold_isAt=FALSE
[3/1498] pred_at=PROBABLE gold_at=TRUE | pred_isAt=TRUE gold_isAt=FALSE
[4/1498] pred_at=FALSE gold_at=PROBABLE | pred_isAt=FALSE gold_isAt=FALSE
[5/1498] pred_at=FALSE gold_at=PROBABLE | pred_isAt=FALSE gold_isAt=FALSE
[6/1498] pred_at=FALSE gold_at=PROBABLE | pred_isAt=FALSE gold_isAt=FALSE
[7/1498] pred_at=FALSE gold_at=FALSE | pred_isAt=FALSE gold_isAt=FALSE
[8/1498] pred_at=PROBABLE gold_at=FALSE | pred_isAt=FALSE gold_isAt=FALSE
[9/1498] pred_at=FALSE gold_at=FALSE | pred_isAt=FALSE gold_isAt=FALSE
[10/1498] pred_at=FALSE gold_at=FALSE | pred_isAt=FALSE gold_isAt=FALSE
[11/1498] pred_at=FALSE gold_at=FALSE | pred_isAt=FALSE gold_isAt=FALSE
[12/1498] pred_at=FALSE gold_at=PROBABLE | pred_isAt=FALSE gold_isAt=FALSE
[13/1498] pred_at=FALSE gold_at=PROBABLE | pred_isAt=FALSE gold_isAt

In [ ]:
!cd {drive_repo_path} && python3 scripts/file_scorer_evaluation.py --gold_data_file data/sandbox/{lang}-dev.jsonl --predictions_file /content/{out}


Evaluation Results for qwen_qwen3_32b_RETRIEVAL_WORD_fr.jsonl:
  'at': macro_recall=0.4848, accuracy=0.6455 (967/1498)
  'isAt': macro_recall=0.6376, accuracy=0.8271 (1239/1498)
  'global': macro_recall=0.5612 (2206/2996)



In [ ]:
import shutil
import os

configs = [
    ("qwen3_32b", "qwen3-32b"),
    ("qwen3_14b", "qwen3-14b"),
    ("qwen3_8b", "qwen3-8b"),
]
languages = ["en", "fr", "de"]
shots = ["ZEROSHOT", "4SHOT"]
variants = ["", "_uncleaned"]  # cleaned (default) and uncleaned

copied = 0
skipped = 0

for tag, _ in configs:
    for lang in languages:
        for shot in shots:
            for variant in variants:
                filename = f"qwen_{tag}_{shot}_{lang}{variant}.jsonl"
                if os.path.exists(filename):
                    shutil.copy(filename, f"{drive_repo_path}/{filename}")
                    print(f"✅ Copied: {filename}")
                    copied += 1
                else:
                    print(f"⏭️  Skipped (not found): {filename}")
                    skipped += 1

print(f"\nDone! Copied {copied}, skipped {skipped}")